In [54]:
# imports

import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
from pprint import pprint

In [96]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
MODEL = "llama3.2"
MODEL2 = "deepseek-r1:1.5b"
MODEL3 = "qwen3.5:0.8b"

In [68]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a assistant that analyzes the contents of a website,
and provides a short, ignoring text that might be navigation related but it is conference or events.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [69]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website, access to articles and provide a short summary, events and dates.
If it includes news or announcements, then summarize these too.

"""

In [95]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = MODEL,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [71]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [100]:
url_website = 'https://www.3gpp.org/news-events/conferences-events'
# display_summary(url_website)

In [73]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [98]:
# print(get_links_user_prompt("https://www.3gpp.org/news-events/conferences-events"))

In [94]:
def select_relevant_links(url):
    """ model check with 'ollama list'"""
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [99]:
# select_relevant_links("https://www.3gpp.org/news-events/conferences-events")

# Select all relevant and integrate both calls to LLM - Making the Brochure

In [77]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [78]:
url = 'https://www.3gpp.org/news-events/conferences-events'
name = '3gpp'
# print(fetch_page_and_all_relevant_links("https://www.3gpp.org/news-events/conferences-events")) <-- sometimes return errors.
# contents = fetch_website_contents(url)
# relevant_links = select_relevant_links(url)
# result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
# for link in relevant_links['links']:
#         result += f"\n\n### Link: {link['type']}\n"
#         result += fetch_website_contents(link["url"])
# result

In [79]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [81]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company or organization called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [97]:
# get_brochure_user_prompt(name, url)

In [92]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure(name, url)